In [1]:
import numpy as np
import pickle

from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.preprocessing import normalize
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


import os
import sys
sys.path.append("/home/rguo_hpc/myfolder/mocap")
from retrain.swav.layers import ProjectionHead, PrototypeLayer
from retrain.swav.utils import sinkhorn, swav_loss

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
# Load
mae_tr  = np.load("/home/rguo_hpc/myfolder/mocap/outputs/50patch3/representations/mae_sdannce_tr.npy")[:,25:4525] # 360, 4500, 192
mae_val = np.load("/home/rguo_hpc/myfolder/mocap/outputs/50patch3/representations/mae_sdannce_val.npy")[:,25:4525]
mae_feats = np.concatenate([mae_tr, mae_val]) # 480, 4500, 192
num_seq, L, D = mae_feats.shape

In [3]:
# Flatten + Normalize
mae_feats = mae_feats.reshape(-1, D) # (2160000, 128)
mae_feats_norm = normalize(mae_feats)

In [ ]:
"""
# Normalize by each sequence
mean = mae_feats.mean(axis=(0, 1), keepdims=True)
std = mae_feats.std(axis=(0, 1), keepdims=True)
mae_feats_norm = (mae_feats - mean) / (std + 1e-8)

# PCA
mae_pca = PCA(n_components=128, random_state=42)
mae_pca_feats = mae_pca.fit_transform(mae_feats_norm)
feats_cumvar = np.cumsum(mae_pca.explained_variance_ratio_)
print(f"Variance explained by PCs: {feats_cumvar[-1]:.1%}")
"""

In [ ]:
# Generate 2 views ( or by setting view_invariant?)
feats = mae_feats_norm

view1 = feats[4::9]
view2_selected = []
for start in range(0, len(feats), 9):
    end = min(start + 9, len(feats))
    candidates = np.arange(start, end)
    candidates = candidates[candidates != start + 4]  # exclude 5th element
    view2_selected.append(np.random.choice(candidates))
view2 = feats[view2_selected] 

# shuffle
idx = np.random.permutation(len(view1))
view1_shuffled = view1[idx]
view2_shuffled = view2[idx]
print(view1_shuffled.shape)
print(view2_shuffled.shape)

(240000, 192)
(240000, 192)


In [7]:
#arr1 = mae_pca_feats[0::9]
#arr2 = mae_pca_feats[8::9]
#np.save("view1_shuffled.npy", view1_shuffled)
#np.save("view2_shuffled.npy", view2_shuffled)

# Load

In [17]:
K = 128
sample_freq = 9

In [ ]:
# load labels
with open("/home/rguo_hpc/myfolder/data/sdannce/data_fmr1.pkl", 'rb') as file:
    data_fmr1 = pickle.load(file)
fmr1_fold_1 = {"train":[402, 404, 405, 406, 407, 408], "valid": [401, 403]} # In total 8 mice, each 3 sequnces with 90000 frames

hlac_labels = []
for mouse in fmr1_fold_1["train"]+fmr1_fold_1["valid"]:
    num_seq = len(data_fmr1[mouse]["hlac"])
    for i in range(num_seq):
        #data_fmr1[401]["ratgen"] [1,1,1]
        hlac = np.squeeze(data_fmr1[mouse]["hlac"][i])
        hlac_labels.append(hlac)

hlac_labels = np.array(hlac_labels).reshape(-1)[::sample_freq]
hlac_tr = hlac_labels[:180000]
hlac_val = hlac_labels[180000:]
print(hlac_tr.shape)
print(hlac_val.shape)

(180000,)
(60000,)


In [106]:
"""
# K-means for init
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(gmm.means_)
centers = kmeans.cluster_centers_
"""
# GMM for init when projection head is None
gmm = GaussianMixture(n_components=K, covariance_type="diag", random_state=42, n_init=3)
gmm.fit(feats[::9])
centers = gmm.means_

In [ ]:
# --------------------------------------------------------------------------
# 1. Dataset over precomputed paired embeddings
# --------------------------------------------------------------------------
class PairedEmbeddingDataset(Dataset):
    def __init__(self, view_a_path: str, view_b_path: str):
        self.z_a = np.load(view_a_path)
        self.z_b = np.load(view_b_path)
        assert self.z_a.shape == self.z_b.shape, (f"view shapes must match: {self.z_a.shape} vs {self.z_b.shape}")
        self.z_a = torch.from_numpy(self.z_a).float()
        self.z_b = torch.from_numpy(self.z_b).float()
    def __len__(self):
        return self.z_a.shape[0]
    def __getitem__(self, idx):
        return self.z_a[idx], self.z_b[idx]

# --------------------------------------------------------------------------
# 2. Training loop (prototypes only, no encoder)
# --------------------------------------------------------------------------
def train_prototypes(view_a_path: str, view_b_path: str, num_prototypes: int, batch_size: int, num_epochs: int, 
                     lr: float, weight_decay: float, freeze_prototypes_epoch0: bool = True, gmm_means: np.ndarray = None,
                     device: str = "cuda" if torch.cuda.is_available() else "cpu", log_every: int = 50, 
                     projection_hidden_dim: int = 192, projection_out_dim: int = None,):
    
    dataset = PairedEmbeddingDataset(view_a_path, view_b_path)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    embed_dim = dataset.z_a.shape[1]
    
    projection_head = None
    proto_in_dim = embed_dim
    if projection_out_dim is not None:
        projection_head = ProjectionHead(embed_dim, projection_hidden_dim, projection_out_dim).to(device)
        proto_in_dim = projection_out_dim
    
    prototypes = PrototypeLayer(embed_dim, num_prototypes).to(device)
    
    if gmm_means is not None: # warm-start from your existing GMM cluster centers, if you have them
        # NOTE: gmm_means must be in the same space the prototypes operate on — i.e. in projection_out_dim space 
        # if you're using a head, not raw embed_dim space. If your GMM was fit on the raw frozen embeddings, 
        # ONLY use this warm-start when projection_out_dim=None.
        centers = torch.as_tensor(gmm_means, dtype=torch.float32)
        prototypes.init_from_centers(F.normalize(centers, dim=1, p=2))

    trainable_params = list(prototypes.parameters())
    if projection_head is not None:
        trainable_params += list(projection_head.parameters())

    optimizer = torch.optim.AdamW(prototypes.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(num_epochs):
        running_loss = 0.0
        for step, (z_a, z_b) in enumerate(loader):
            z_a, z_b = z_a.to(device), z_b.to(device)
            if projection_head is not None:
                p_a = projection_head(z_a)
                p_b = projection_head(z_b)
            else:
                p_a, p_b = z_a, z_b
 
            p_a = F.normalize(p_a, dim=1, p=2)
            p_b = F.normalize(p_b, dim=1, p=2)
            scores_a = prototypes(z_a)
            scores_b = prototypes(z_b)

            loss = swav_loss(scores_a, scores_b)
            optimizer.zero_grad()
            loss.backward()
            if freeze_prototypes_epoch0 and epoch == 0:
                for p in prototypes.parameters():
                    p.grad = None
            optimizer.step()
            prototypes.normalize_prototypes()
            running_loss += loss.item()
            if step % log_every == 0:
                print(f"epoch {epoch:3d}  step {step:4d}  loss {loss.item():.4f}")
        print(f"epoch {epoch:3d} done, avg loss {running_loss / len(loader):.4f}")

    return prototypes, projection_head, dataset

In [22]:
VIEW_A_PATH = "view1_shuffled.npy"   # <-- adjust paths
VIEW_B_PATH = "view2_shuffled.npy"
NUM_PROTOTYPES = K         # <-- match your expected number of behaviors
NUM_EPOCHS = 10
BATCH_SIZE = 1024
PROJECTION_HIDDEN_DIM = 256
PROJECTION_OUT_DIM = 128
lr = 5e-4
weight_decay = 2e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBED_DIM = 192

prototypes, projection_head, dataset = train_prototypes(
                    VIEW_A_PATH, VIEW_B_PATH, num_prototypes=NUM_PROTOTYPES, 
                    batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS, 
                    projection_hidden_dim = PROJECTION_HIDDEN_DIM, projection_out_dim=PROJECTION_OUT_DIM,
                    lr = lr, weight_decay=weight_decay) # gmm_means = centers

torch.save(prototypes.state_dict(), "trained_prototypes.pt")
if projection_head is not None:
    torch.save(projection_head.state_dict(), "trained_projection_head.pt")
    print("Saved trained_projection_head.pt")

epoch   0  step    0  loss 4.7012
epoch   0  step   50  loss 4.4751
epoch   0  step  100  loss 4.4723
epoch   0  step  150  loss 4.4833
epoch   0  step  200  loss 4.4779
epoch   0 done, avg loss 4.4794
epoch   1  step    0  loss 4.4783
epoch   1  step   50  loss 2.8718
epoch   1  step  100  loss 2.3327
epoch   1  step  150  loss 2.0601
epoch   1  step  200  loss 1.9791
epoch   1 done, avg loss 2.4919
epoch   2  step    0  loss 1.9301
epoch   2  step   50  loss 1.8617
epoch   2  step  100  loss 1.8541
epoch   2  step  150  loss 1.8652
epoch   2  step  200  loss 1.8712
epoch   2 done, avg loss 1.8823
epoch   3  step    0  loss 1.8523
epoch   3  step   50  loss 1.8222
epoch   3  step  100  loss 1.8444
epoch   3  step  150  loss 1.8454
epoch   3  step  200  loss 1.8674
epoch   3 done, avg loss 1.8470
epoch   4  step    0  loss 1.8899
epoch   4  step   50  loss 1.8549
epoch   4  step  100  loss 1.8508
epoch   4  step  150  loss 1.8518
epoch   4  step  200  loss 1.8470
epoch   4 done, avg lo

In [ ]:
# --------------------------------------------------------------------------
# 3. Compute new representations using the trained prototypes
# --------------------------------------------------------------------------
@torch.no_grad()
def compute_new_representations(prototypes: PrototypeLayer, embeddings: torch.Tensor, device: str,
                                projection_head: nn.Module = None, which: str = "cluster",
                                temperature: float = 0.1, batch_size: int = 1024,):
    """
    embeddings: (N, D) raw frame embeddings (NOT yet L2-normalized)
    which:
      "cluster"    - (N, K) softmax-over-prototypes soft assignment. Always available. This is the only new signal you get 
                    if projection_head is None (prototypes-only training).
      "projection" - (N, D') the projection head's output (L2-normalized), i.e. an actual new learned embedding. 
                    Requires projection_head is not None.
    """
    prototypes = prototypes.to(device)
    prototypes.eval()
    if projection_head is not None:
        projection_head = projection_head.to(device)
        projection_head.eval()

    out = []
    for i in range(0, embeddings.shape[0], batch_size):
        chunk = embeddings[i:i + batch_size].to(device)
        p = projection_head(chunk) if projection_head is not None else chunk
        z = F.normalize(p, dim=1, p=2)
        if which == "projection":
            out.append(z.cpu())
        elif which == "cluster":
            scores = prototypes(z)
            probs = F.softmax(scores / temperature, dim=1)
            out.append(probs.cpu())
    return torch.cat(out, dim=0)

In [ ]:
# After training
# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
prototypes =  PrototypeLayer(embed_dim=EMBED_DIM, num_prototypes=NUM_PROTOTYPES)
checkpoint_model = torch.load("trained_prototypes.pt", map_location=device, weights_only=False)
prototypes.load_state_dict(checkpoint_model, strict=True)

which = "projection" if PROJECTION_OUT_DIM is not None else "cluster"
projection_head =  ProjectionHead(in_dim=EMBED_DIM, hidden_dim=PROJECTION_HIDDEN_DIM, out_dim=PROJECTION_OUT_DIM)
checkpoint = torch.load("trained_projection_head.pt", map_location=device, weights_only=False)
projection_head.load_state_dict(checkpoint, strict=True)


# Compute the new K-dim representation for every frame in view_a. (Using view_a as "the" per-frame representation 
# since view_b was only a temporal-pairing signal for training, not a separate population of frames you necessarily 
# want represented — adjust if your two files actually cover different/non-overlapping frames you want stacked.)
new_repr = compute_new_representations(prototypes, torch.from_numpy(feats[::sample_freq]).float(), device, 
                                       projection_head = projection_head, which="projection",
                                       #projection_head = None, which="cluster"
                                      )
mae_feats_tr = new_repr[:180000,]
mae_feats_val = new_repr[180000:,]

# If you'd rather keep the original D-dim embedding AND add the cluster representation (rather than replace it), concatenate instead:
# combined = torch.cat([F.normalize(dataset.z_a, dim=1), new_repr], dim=1)

In [25]:
# model
model = LogisticRegression(max_iter=500, multi_class='multinomial')
#model = RandomForestClassifier()
# fit & predict
model.fit(mae_feats_tr,  hlac_tr)

y_pred = model.predict(mae_feats_val)
print("Accuracy:", accuracy_score(hlac_val, y_pred))
print("\nClassification Report:\n", classification_report(hlac_val, y_pred))

/home/rguo_hpc/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.5631

Classification Report:
               precision    recall  f1-score   support

           1       0.56      0.31      0.40     11792
           2       0.48      0.62      0.54     16949
           3       0.64      0.64      0.64      2183
           4       0.71      0.54      0.61      2158
           5       0.26      0.01      0.01       961
           6       0.82      0.96      0.89      6165
           7       0.48      0.62      0.54     12772
           8       0.77      0.45      0.57      7020

    accuracy                           0.56     60000
   macro avg       0.59      0.52      0.53     60000
weighted avg       0.58      0.56      0.55     60000

